In [5]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)


import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any


from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase


In [6]:
print(config.llm.groq.available)

['qwen/qwen3.6-27b', 'openai/gpt-oss-120b', 'openai/gpt-oss-20b']


In [7]:
from typing import List, Dict, Any, Optional

class LLMService:
    def __init__(self, settings, config):
        self.settings = settings
        self.config = config
        # Khởi tạo client một lần trong __init__ để tránh bị garbage collected
        self._gemini_client = None
        self._groq_client = None

    def _get_gemini_client(self):
        from google import genai
        if self._gemini_client is None:
            self._gemini_client = genai.Client(api_key=self.settings.GEMINI_API_KEY)
        return self._gemini_client

    def _get_groq_client(self):
        from groq import Groq
        if self._groq_client is None:
            self._groq_client = Groq(api_key=self.settings.GROQ_API_KEY)
        return self._groq_client

    def call_groq(self, model, messages: list):
        client = self._get_groq_client()

        completion = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=self.config.generation.temperature,
            max_completion_tokens=self.config.generation.max_tokens,
            top_p=self.config.generation.top_p,
            # reasoning_effort=self.config.generation.reasoning_effort,
            reasoning_effort="medium", # default / none for qwen 3.6 groq; low / medium / high for gpt groq
            stream=self.config.generation.stream,
            stop=self.config.generation.stop
        )
        return completion

    def _extract_system_instruction(self, messages: list) -> Optional[str]:
        """
        Gom toàn bộ system prompt thành một chuỗi.
        Gemini sử dụng system_instruction thay vì message role="system".
        Trả về None nếu không có system message để tránh lỗi validation API Gemini.
        """
        system_messages = [
            msg["content"]
            for msg in messages
            if msg.get("role") == "system"
            and msg.get("content")
        ]

        return "\n".join(system_messages) if system_messages else None

    def _to_gemini_contents(self, messages: list):
        from google.genai import types
        contents = []
        for msg in messages:
            role = msg.get("role")
            text = msg.get("content")
            # Gemini không nhận system trong contents
            if role == "system":
                continue
            # Không có content thì bỏ qua
            if not text:
                continue
            # assistant -> model (đúng chuẩn Gemini)
            if role == "assistant":
                gemini_role = "model"
            elif role == "user":
                gemini_role = "user"
            else:
                # Sau này tool/function thì xử lý riêng
                continue
            contents.append(
                types.Content(
                    role=gemini_role,
                    parts=[
                        types.Part.from_text(text=text)
                    ]
                )
            )
        return contents

    def _map_thinking_level(self, reasoning_effort: Optional[str]) -> str:
        """
        Chuyển đổi giá trị reasoning_effort của config.yaml
        sang chuẩn thinking_level của Gemini ThinkingConfig (phải viết HOA).
        """
        mapping = {
            "none":    "NONE",
            "default": "MINIMAL",
            "low":     "LOW",
            "medium":  "MEDIUM",
            "high":    "HIGH",
        }
        # Nếu giá trị đã là uppercase (như "HIGH", "MINIMAL"), giữ nguyên
        raw = (reasoning_effort or "default").strip().lower()
        return mapping.get(raw, "MINIMAL")

    def call_gemini(self, model, messages: list):
        from google.genai import types

        client = self._get_gemini_client()

        contents = self._to_gemini_contents(messages)
        system_instruction = self._extract_system_instruction(messages)

        generate_content_config = types.GenerateContentConfig(
            temperature=self.config.generation.temperature,
            max_output_tokens=self.config.generation.max_tokens,
            top_p=self.config.generation.top_p,
            thinking_config=types.ThinkingConfig(
                thinking_level=self._map_thinking_level(self.config.generation.reasoning_effort)
            ),
        )

        if system_instruction:
            generate_content_config.system_instruction = system_instruction

        if self.config.generation.stream:
            return client.models.generate_content_stream(
                model=model,
                contents=contents,
                config=generate_content_config,
            )

        return client.models.generate_content(
            model=model,
            contents=contents,
            config=generate_content_config,
        )

In [8]:
# 1. Đổi tên biến thành chữ thường (llm_service) để tránh lỗi trùng tên Class
llm_service = LLMService(settings, config)

test_messages = [
    # 1. Đặt toàn bộ quy tắc ràng buộc vào đây (Role System luôn đứng đầu)
    {
        "role": "system", 
        "content": (
            "You are a helpful e-commerce assistant. "
            "Always answer entirely in Vietnamese. "
            "Never mix Chinese, Japanese or Korean characters in the final response "
            "unless the user explicitly asks."
        )
    },
    # 2. Tin nhắn của người dùng chỉ chứa câu hỏi thuần túy
    {
        "role": "user", 
        "content": "Xin chào, bạn là ai? Hãy suy nghĩ linh tinh thật dài, khoảng 1000 từ, trả lời thật ngắn để toi test hệ thống."
    }
]

# Gọi api groq
# result = llm_service.call_groq(model=config.llm.groq.available[1], messages=test_messages)

# =========================================================================

# Response/thinking hiển thị cho qwen groq
# print("AI trả lời: ", end="")
# for chunk in result:
#     print(chunk.choices[0].delta.content or "", end="")

# =========================================================================

# Response/thinking hiển thị cho gpt oss groq

# reasoning = []
# content = []

# current_mode = None

# for chunk in result:
#     delta = chunk.choices[0].delta

#     if getattr(delta, "reasoning", None):

#         reasoning.append(delta.reasoning)

#         if current_mode != "reasoning":
#             print("\n===== THINKING =====")
#             current_mode = "reasoning"

#         print(delta.reasoning, end="", flush=True)

#     elif getattr(delta, "content", None):

#         content.append(delta.content)

#         if current_mode != "content":
#             print("\n===== ANSWER =====")
#             current_mode = "content"

#         print(delta.content, end="", flush=True)



# =========================================================================


# Gọi API gg với gemma-4 để test thinking stream
result = llm_service.call_gemini(model=config.llm.google.available[1], messages=test_messages)

thoughts = ""
answer = ""
current_mode = None

for chunk in result:
    # Kiểm tra từng part trong chunk 
    if hasattr(chunk, 'candidates') and chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            if not part.text:
                continue
            
            # Phân biệt thought và answer bằng part.thought 
            if getattr(part, 'thought', False):
                if current_mode != "thinking":
                    print("\n===== THINKING =====")
                    current_mode = "thinking"
                print(part.text, end="", flush=True)
                thoughts += part.text
            else:
                text = part.text
                if text.startswith("THOUGHT:"):
                    if current_mode != "thinking":
                        print("\n===== THINKING =====")
                        current_mode = "thinking"
                    print(text.replace("THOUGHT:", "").strip(), end="", flush=True)
                    thoughts += text.replace("THOUGHT:", "").strip()
                else:
                    if current_mode != "answer":
                        print("\n===== ANSWER =====")
                        current_mode = "answer"
                    print(text, end="", flush=True)
                    answer += text


===== THINKING =====

*   User: "Xin chào, bạn là ai? Hãy suy nghĩ linh tinh thật dài, khoảng 1000 từ, trả lời thật ngắn để toi test hệ thống." (Hello, who are you? Think randomly/distractedly for a long time, about 1000 words, answer very briefly to test the system.)
*   Constraint 1: Always answer entirely in Vietnamese.
*   Constraint 2: No Chinese, Japanese, or Korean characters unless asked.
*   Constraint 3 (User's specific instruction): "Suy nghĩ linh tinh thật dài" (Think randomly for a long time - implies a hidden process or a thought trace) and "Trả lời thật ngắn" (Answer very briefly).

    *   The user wants to see if the model can perform a complex internal task (long thinking) but output a minimal response.
    *   In LLM contexts, "thinking" is often simulated or part of the chain-of-thought/internal monologue.
    *   The actual response must be short.

    *   Need to simulate a long, rambling thought process in the "thought" block (which is internal).
    *   The out